In [ ]:
%%sql -r dataframe_1
use role sysadmin;
use warehouse compute_wh;
use schema legacy_db.raw_sch;

****
### Create tables inside the raw layer
****

In [ ]:
%%sql -r dataframe_3
create or replace table legacy_db.raw_sch.customers_raw(
    cust_key number,
    name text,
    address text,
    nation_name text,
    phone text,
    acct_bal number,
    mkt_segment text,
    load_ts timestamp,
    load_row_number number,
    load_file_name text
);

****
### Creating stream on customer_raw to track the changes
****

In [ ]:
%%sql -r dataframe_5
create or replace stream legacy_db.raw_sch.customer_raw_stream
    on table legacy_db.raw_sch.customers_raw
    append_only = true;

****
### Creating order table with 11 columns
****

In [ ]:
%%sql -r dataframe_7
create or replace table legacy_db.raw_sch.orders_raw(
    order_key number,
    cust_key number,
    order_status text(1),
    total_price number,
    order_date date,
    order_priority text,
    clerk text,
    ship_priority number(1),
    load_ts timestamp,
    load_row_number number,
    load_file_name text
);

****
### Creating Stream for Orders Table
****

In [ ]:
%%sql -r dataframe_9
create or replace stream legacy_db.raw_sch.orders_raw_stream
    on table legacy_db.raw_sch.orders_raw
    append_only = true;

****
### Run copy command and check data set
****

In [ ]:
%%sql -r dataframe_11
create or replace task legacy_db.raw_sch.root_task
    warehouse = compute_wh
    schedule = '1 minute'
    as select current_role();

In [ ]:
%%sql -r dataframe_12
create or replace task legacy_db.raw_sch.copy_to_customer_raw_task
warehouse = compute_wh
after legacy_db.raw_sch.root_task
as
copy into legacy_db.raw_sch.customers_raw from
(
    select t.$1,t.$2,t.$3,t.$4,t.$5,t.$6,t.$7,
        current_timestamp(),
        metadata$file_row_number,
        metadata$filename
    from @legacy_db.source_sch.my_stage/customer/ as t
) file_format = (format_name = 'legacy_db.source_sch.csv_format');

In [ ]:
%%sql -r dataframe_14
create or replace task legacy_db.raw_sch.copy_to_order_raw_task
warehouse = compute_wh
after legacy_db.raw_sch.root_task
as
copy into legacy_db.raw_sch.orders_raw from
(
select 
        t.$1,t.$2,t.$3,t.$4,t.$5,t.$6,t.$7,t.$8,
        current_timestamp(),
        metadata$file_row_number,
        metadata$filename
    from @legacy_db.source_sch.my_stage/order/ as t
) file_format = (format_name = 'legacy_db.source_sch.csv_format');